In [9]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# 1. 데이터 로드 및 클래스 확인
image_size = (150, 150)
batch_size = 32

print("데이터 로드 중...")
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "./train", image_size=image_size, batch_size=batch_size)
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "./valid", image_size=image_size, batch_size=batch_size)
test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "./test", image_size=image_size, batch_size=batch_size)

# Keras가 폴더명으로 자동 생성한 클래스 이름 확인
class_names = train_ds.class_names
print(f"클래스 분류 기준: {class_names}")

# 속도 최적화
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(lambda x, y: (x, tf.expand_dims(tf.cast(y, tf.float32), -1)))
val_ds   = val_ds.map(lambda x, y: (x, tf.expand_dims(tf.cast(y, tf.float32), -1)))
test_ds  = test_ds.map(lambda x, y: (x, tf.expand_dims(tf.cast(y, tf.float32), -1)))

# 2. 데이터 증강 및 모델 설계
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

model = keras.Sequential([
    layers.Input(shape=(150, 150, 3)),
    data_augmentation, 
    layers.Rescaling(1./255), 

    layers.Conv2D(32, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation='relu'),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid') # 0(no) 또는 1(yes)
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy", threshold=0.5),
        tf.keras.metrics.Precision(name="precision", thresholds=0.5),
        tf.keras.metrics.Recall(name="recall", thresholds=0.5),
        tf.keras.metrics.F1Score(name="f1", threshold=0.5),
    ],
)

# 3. 콜백(Callbacks) 설정 (핵심 추가 사항)
# 3-1. EarlyStopping: valid 정확도가 5번(patience)의 에포크 동안 안 오르면 학습 강제 종료
early_stopping = EarlyStopping(
    monitor='val_accuracy', 
    patience=5, 
    restore_best_weights=True # 가장 좋았던 가중치로 복구
)

# 3-2. ModelCheckpoint: valid 정확도가 갱신될 때마다 최고 성능의 모델을 파일로 저장
model_checkpoint = ModelCheckpoint(
    filepath='best_yes_no_model.keras', # 저장될 파일 이름
    monitor='val_accuracy',
    save_best_only=True
)

# 4. 학습 진행
print("\n학습을 시작합니다...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50, # EarlyStopping이 있으므로 에포크를 넉넉하게 50으로 잡습니다.
    callbacks=[early_stopping, model_checkpoint] # 콜백 적용
)

# 5. 최종 성능 평가
outs = model.evaluate(test_ds)
for name, val in zip(model.metrics_names, outs):
    print(f"{name}: {val}")

Found 5216 files belonging to 2 classes.
Found 16 files belonging to 2 classes.
Found 624 files belonging to 2 classes.
클래스 분류 기준: ['NORMAL', 'PNEUMONIA']

Epoch 1/50
163/163 ━━━━━━━━━━━━━━━━━━━━ 15s 77ms/step - accuracy: 0.7703 - f1: 0.8605 - loss: 0.5207 - precision: 0.7841 - recall: 0.9533 - val_accuracy: 0.6250 - val_f1: 0.7000 - val_loss: 0.5837 - val_precision: 0.5833 - val_recall: 0.8750
Epoch 2/50
163/163 ━━━━━━━━━━━━━━━━━━━━ 12s 73ms/step - accuracy: 0.8604 - f1: 0.9069 - loss: 0.3361 - precision: 0.8989 - recall: 0.9151 - val_accuracy: 0.7500 - val_f1: 0.7778 - val_loss: 0.5939 - val_precision: 0.7000 - val_recall: 0.8750
Epoch 3/50
163/163 ━━━━━━━━━━━━━━━━━━━━ 12s 70ms/step - accuracy: 0.8894 - f1: 0.9260 - loss: 0.2777 - precision: 0.9207 - recall: 0.9314 - val_accuracy: 0.6250 - val_f1: 0.7273 - val_loss: 1.2020 - val_precision: 0.5714 - val_recall: 1.0000
Epoch 4/50
163/163 ━━━━━━━━━━━━━━━━━━━━ 11s 69ms/step - accuracy: 0.9055 - f1: 0.9369 - loss: 0.2413 - precision: 0.92